In [ ]:
# Initialize Earth Engine
import ee
try:
    ee.Initialize(project='river-468515')
    print("Earth Engine initialized successfully!")
except:
    ee.Authenticate()
    ee.Initialize(project='river-468515')
    print("Earth Engine authenticated and initialized!")

Earth Engine authenticated and initialized!


In [ ]:
# ======================================================================
# Sentinel Quarterly Water Mask Export - 2015-2025 (FIXED)
# ======================================================================

import os
import ee
import geemap
import time
import numpy as np

# ======================================================================
#                            CONFIGURATION
# ======================================================================

study_area = ee.Geometry.Polygon([
    [88.75677098777692, 24.00347653384235],
    [90.59972753074567, 23.152644569433964],
    [90.52556981590192, 23.54602121781216],
    [88.74480244337559, 24.334092333430064],
    [88.75677098777692, 24.00347653384235]
])

START_YEAR = 2015
END_YEAR = 2025

out_dir = "./Sentinel_Quarterly_WaterMasks_2015-2025"
os.makedirs(out_dir, exist_ok=True)

WATER_THRESHOLD = 0
CLOUD_COVER_MAX = 50
EXPORT_SCALE = 60  # Match Landsat resolution for consistency

# ⭐ NEW: IMAGE LIMITS TO PREVENT TIMEOUT
MAX_S2_IMAGES = 25  # Limit Sentinel-2 images (best quality ones)
MAX_S1_IMAGES = 20  # Limit Sentinel-1 images

QUARTERS = {
    'Q1': {'months': [1, 2, 3], 'name': 'Jan-Mar'},
    'Q2': {'months': [4, 5, 6], 'name': 'Apr-Jun'},
    'Q3': {'months': [7, 8, 9], 'name': 'Jul-Sep'},
    'Q4': {'months': [10, 11, 12], 'name': 'Oct-Dec'}
}

# ======================================================================
#                         HELPER FUNCTIONS
# ======================================================================

def mask_sentinel2_clouds(image):
    """Mask clouds in Sentinel-2 imagery using QA60 band"""
    qa = image.select('QA60')
    # Bits 10 and 11 are clouds and cirrus
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000)

def add_sentinel1_ratio(image):
    """Add VV/VH ratio band for Sentinel-1"""
    vv = image.select('VV')
    vh = image.select('VH')
    ratio = vv.divide(vh).rename('VV_VH_ratio')
    return image.addBands(ratio)

def export_quarterly_water_mask(year, quarter, quarter_months, quarter_name):
    """Export quarterly water mask using Sentinel-1 and Sentinel-2"""
    try:
        start_month = quarter_months[0]
        end_month = quarter_months[-1]

        start_date = ee.Date.fromYMD(year, start_month, 1)
        end_date = ee.Date.fromYMD(year, end_month, 1).advance(1, 'month')

        # ==============================================================
        # SENTINEL-2 Processing (Optical - primary for water detection)
        # ⭐ FIXED: Sort by cloud cover and limit to best images
        # ==============================================================
        s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
            .filterDate(start_date, end_date) \
            .filterBounds(study_area) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER_MAX)) \
            .sort('CLOUDY_PIXEL_PERCENTAGE') \
            .limit(MAX_S2_IMAGES) \
            .map(mask_sentinel2_clouds) \
            .select(['B3', 'B11'], ['Green', 'SWIR1'])

        s2_count = s2_collection.size().getInfo()

        # ==============================================================
        # SENTINEL-1 Processing (SAR - supplementary)
        # ⭐ FIXED: Sort by date (most recent) and limit images
        # ==============================================================
        s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterDate(start_date, end_date) \
            .filterBounds(study_area) \
            .filter(ee.Filter.eq('instrumentMode', 'IW')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
            .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
            .sort('system:time_start', False) \
            .limit(MAX_S1_IMAGES) \
            .select(['VV', 'VH']) \
            .map(add_sentinel1_ratio)

        s1_count = s1_collection.size().getInfo()

        # Check if we have data
        if s2_count == 0 and s1_count == 0:
            return {
                'year': year,
                'quarter': quarter,
                'water_area_km2': 0,
                'filename': 'N/A',
                'status': 'no_images',
                'image_count': 0,
                's2_count': 0,
                's1_count': 0
            }

        # ==============================================================
        # Primary: Sentinel-2 MNDWI approach
        # ==============================================================
        water_mask = None
        method_used = ''

        if s2_count > 0:
            # Create Sentinel-2 composite
            s2_composite = s2_collection.median().clip(study_area)

            # Calculate MNDWI (Modified Normalized Difference Water Index)
            mndwi = s2_composite.normalizedDifference(['Green', 'SWIR1'])
            water_mask = mndwi.gt(WATER_THRESHOLD).rename('water').toByte()
            method_used = 'S2_MNDWI'

        # ==============================================================
        # Fallback: Sentinel-1 SAR approach if S2 insufficient
        # ==============================================================
        elif s1_count > 0:
            # Create Sentinel-1 composite
            s1_composite = s1_collection.median().clip(study_area)

            # Water detection using VV polarization (water has low backscatter)
            # Threshold typically around -16 to -18 dB for water
            vv = s1_composite.select('VV')
            water_mask = vv.lt(-16).rename('water').toByte()
            method_used = 'S1_VV'

        if water_mask is None:
            return {
                'year': year,
                'quarter': quarter,
                'water_area_km2': 0,
                'filename': 'N/A',
                'status': 'processing_failed',
                'image_count': 0,
                's2_count': s2_count,
                's1_count': s1_count
            }

        # Calculate water area
        pixel_area = water_mask.multiply(ee.Image.pixelArea())
        water_area = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=study_area,
            scale=30,
            maxPixels=1e10,
            bestEffort=True
        )

        area_km2 = ee.Number(water_area.get('water', 0)).divide(1e6).getInfo()

        # Skip if no water detected
        if area_km2 == 0:
            return {
                'year': year,
                'quarter': quarter,
                'water_area_km2': 0,
                'filename': 'N/A',
                'status': 'no_water',
                'image_count': s2_count + s1_count,
                's2_count': s2_count,
                's1_count': s1_count
            }

        # Export with coarser resolution (matching Landsat)
        out_tif = os.path.join(out_dir, f"water_mask_{year:04d}_{quarter}.tif")

        # ⭐ FIXED: Use more robust export with retries
        export_success = False

        # Try 1: Standard export
        try:
            geemap.ee_export_image(
                water_mask.selfMask(),
                filename=out_tif,
                scale=EXPORT_SCALE,
                region=study_area,
                file_per_band=False,
                crs='EPSG:4326',
                timeout=300  # 5 minute timeout
            )
            if os.path.exists(out_tif):
                export_success = True
        except Exception as e1:
            print(f"\n  Retry 1 failed: {str(e1)[:50]}", end='')

            # Try 2: Use bounds instead of polygon
            try:
                bounds = study_area.bounds()
                geemap.ee_export_image(
                    water_mask.selfMask(),
                    filename=out_tif,
                    scale=EXPORT_SCALE,
                    region=bounds,
                    file_per_band=False,
                    crs='EPSG:4326',
                    timeout=300
                )
                if os.path.exists(out_tif):
                    export_success = True
            except Exception as e2:
                print(f"\n  Retry 2 failed: {str(e2)[:50]}", end='')

                # Try 3: Increase scale to 120m (coarser, faster)
                try:
                    geemap.ee_export_image(
                        water_mask.selfMask(),
                        filename=out_tif,
                        scale=120,
                        region=study_area,
                        file_per_band=False,
                        crs='EPSG:4326',
                        timeout=300
                    )
                    if os.path.exists(out_tif):
                        export_success = True
                        print(f" (120m)", end='')
                except Exception as e3:
                    pass

        if export_success:
            return {
                'year': year,
                'quarter': quarter,
                'water_area_km2': area_km2,
                'filename': os.path.basename(out_tif),
                'status': 'success',
                'image_count': s2_count + s1_count,
                's2_count': s2_count,
                's1_count': s1_count,
                'method': method_used
            }
        else:
            return {
                'year': year,
                'quarter': quarter,
                'water_area_km2': area_km2,
                'filename': 'N/A',
                'status': 'export_failed',
                'image_count': s2_count + s1_count,
                's2_count': s2_count,
                's1_count': s1_count
            }

    except Exception as e:
        return {
            'year': year,
            'quarter': quarter,
            'water_area_km2': 0,
            'filename': 'N/A',
            'status': f'error: {str(e)[:30]}',
            'image_count': 0,
            's2_count': 0,
            's1_count': 0
        }

# ======================================================================
#                       MAIN PROCESSING
# ======================================================================

print("="*60)
print(f"Sentinel Quarterly Water Mask Export ({START_YEAR}-{END_YEAR})")
print(f"Cloud Cover Max: {CLOUD_COVER_MAX}%")
print(f"Export Scale: {EXPORT_SCALE}m (matching Landsat)")
print(f"⭐ Image Limits: S2={MAX_S2_IMAGES}, S1={MAX_S1_IMAGES}")
print("="*60)

# Generate all periods
all_periods = []
for year in range(START_YEAR, END_YEAR + 1):
    for quarter_id, quarter_info in QUARTERS.items():
        all_periods.append({
            'year': year,
            'quarter': quarter_id,
            'quarter_name': quarter_info['name'],
            'months': quarter_info['months']
        })

total_periods = len(all_periods)
print(f"\nTotal quarters: {total_periods}\n")

# Process each period
export_summary = {
    'successful': 0,
    'failed': 0,
    'total_size_mb': 0,
    'areas': [],
    's2_primary': 0,
    's1_fallback': 0
}

print("="*60)
print("EXPORTING WATER MASKS...")
print("="*60)

for i, period_info in enumerate(all_periods):
    year = period_info['year']
    quarter = period_info['quarter']
    label = f"{year}-{quarter}"

    out_tif = os.path.join(out_dir, f"water_mask_{year:04d}_{quarter}.tif")

    # Skip if exists
    if os.path.exists(out_tif):
        file_size_mb = os.path.getsize(out_tif) / (1024 * 1024)
        export_summary['successful'] += 1
        export_summary['total_size_mb'] += file_size_mb
        print(f"[{i+1}/{total_periods}] {label}... ⏭️ Exists ({file_size_mb:.2f} MB)")
        continue

    print(f"[{i+1}/{total_periods}] {label} ({period_info['quarter_name']})...", end='', flush=True)

    result = export_quarterly_water_mask(
        year,
        quarter,
        period_info['months'],
        period_info['quarter_name']
    )

    if result['status'] == 'success':
        file_size_mb = os.path.getsize(out_tif) / (1024 * 1024)
        export_summary['successful'] += 1
        export_summary['total_size_mb'] += file_size_mb
        export_summary['areas'].append(result['water_area_km2'])

        if result.get('method') == 'S2_MNDWI':
            export_summary['s2_primary'] += 1
        elif result.get('method') == 'S1_VV':
            export_summary['s1_fallback'] += 1

        method_label = result.get('method', 'Unknown')
        print(f" ✅ {method_label} ({result['water_area_km2']:.2f} km², {file_size_mb:.2f} MB, S2:{result['s2_count']} S1:{result['s1_count']})")
    else:
        export_summary['failed'] += 1
        print(f" ❌ {result['status']} (S2:{result.get('s2_count', 0)} S1:{result.get('s1_count', 0)})")

    # ⭐ Increased delay to avoid rate limits
    time.sleep(3)

# Summary
print("\n" + "="*60)
print("COMPLETE!")
print("="*60)
print(f"✅ Successful: {export_summary['successful']}/{total_periods}")
print(f"❌ Failed: {export_summary['failed']}")
print(f"💾 Total size: {export_summary['total_size_mb']:.2f} MB")

if export_summary['successful'] > 0:
    print(f"📊 Avg per file: {export_summary['total_size_mb']/export_summary['successful']:.2f} MB")
    print(f"\n🛰️ Method Distribution:")
    print(f"  • Sentinel-2 (MNDWI): {export_summary['s2_primary']}")
    print(f"  • Sentinel-1 (SAR): {export_summary['s1_fallback']}")

if export_summary['areas']:
    print(f"\n💧 Water Area Statistics:")
    print(f"  • Average: {np.mean(export_summary['areas']):.2f} km²")
    print(f"  • Range: {np.min(export_summary['areas']):.2f} - {np.max(export_summary['areas']):.2f} km²")

print(f"\n📁 Output: {out_dir}")
print("="*60)

# ⭐ NEW: Show which quarters need re-processing
if export_summary['failed'] > 0:
    print("\n" + "="*60)
    print("RE-RUN RECOMMENDATION")
    print("="*60)
    print("To retry failed exports, simply re-run this script.")
    print("It will skip existing files and only process missing ones.")
    print("="*60)

In [ ]:
import shutil
from google.colab import files

# Define folder to zip
folder_to_zip = "/content/Sentinel_Quarterly_WaterMasks_2015-2025"
zip_filename = "/content/Sentinel_Quarterly_WaterMasks_2015-2025.zip"

# Create zip
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', folder_to_zip)

# Download zip
files.download(zip_filename)


Sentinel Bi-Monthly Water Mask Export (2015-2025)

Total bi-monthly periods: 66

[1/66] 2015-BM1 (Jan-Feb)...
Generating URL ...
Please wait ...
Data downloaded to /content/Sentinel_BiMonthly_WaterMasks_2015-2025/water_mask_2015_BM1.tif
  ✅ S1_VV (779.73 km²)
[2/66] 2015-BM2 (Mar-Apr)...
Generating URL ...
Please wait ...
Data downloaded to /content/Sentinel_BiMonthly_WaterMasks_2015-2025/water_mask_2015_BM2.tif
  ✅ S1_VV (604.65 km²)
[3/66] 2015-BM3 (May-Jun)...
Generating URL ...
Please wait ...
Data downloaded to /content/Sentinel_BiMonthly_WaterMasks_2015-2025/water_mask_2015_BM3.tif
  ✅ S1_VV (729.00 km²)
[4/66] 2015-BM4 (Jul-Aug)...
Generating URL ...
Please wait ...
Data downloaded to /content/Sentinel_BiMonthly_WaterMasks_2015-2025/water_mask_2015_BM4.tif
  ✅ S1_VV (886.48 km²)
[5/66] 2015-BM5 (Sep-Oct)...
Generating URL ...
Please wait ...
Data downloaded to /content/Sentinel_BiMonthly_WaterMasks_2015-2025/water_mask_2015_BM5.tif
  ✅ S1_VV (881.85 km²)
[6/66] 2015-BM6 (Nov-Dec

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ======================================================================
# Landsat Bi-Monthly Water Mask Export - Coarser Resolution Method
# ======================================================================

import os
import ee
import geemap
import time
import numpy as np

# ======================================================================
#                            CONFIGURATION
# ======================================================================

study_area = ee.Geometry.Polygon([
    [88.75677098777692, 24.00347653384235],
    [90.59972753074567, 23.152644569433964],
    [90.52556981590192, 23.54602121781216],
    [88.74480244337559, 24.334092333430064],
    [88.75677098777692, 24.00347653384235]
])

START_YEAR = 2000
END_YEAR = 2014

out_dir = "./Landsat_BiMonthly_WaterMasks_2000-2014"
os.makedirs(out_dir, exist_ok=True)

WATER_THRESHOLD = 0
CLOUD_COVER_MAX = 50
EXPORT_SCALE = 60  # Increased from 30 to reduce file size

BIMONTHS = {
    'BM1': {'months': [1, 2], 'name': 'Jan-Feb'},
    'BM2': {'months': [3, 4], 'name': 'Mar-Apr'},
    'BM3': {'months': [5, 6], 'name': 'May-Jun'},
    'BM4': {'months': [7, 8], 'name': 'Jul-Aug'},
    'BM5': {'months': [9, 10], 'name': 'Sep-Oct'},
    'BM6': {'months': [11, 12], 'name': 'Nov-Dec'}
}

# ======================================================================
#                         HELPER FUNCTIONS
# ======================================================================

def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow = 1 << 3
    cloud = 1 << 4
    mask = qa.bitwiseAnd(cloud_shadow).eq(0).And(qa.bitwiseAnd(cloud).eq(0))
    optical_bands = image.select(['Green', 'SWIR1']).multiply(0.0000275).add(-0.2)
    return image.select('QA_PIXEL').addBands(optical_bands, overwrite=True).updateMask(mask)

def export_bimonthly_water_mask(year, bimonth, bimonth_months, bimonth_name):
    """Export bi-monthly water mask with coarser resolution"""
    try:
        start_month = bimonth_months[0]
        end_month = bimonth_months[-1]

        start_date = ee.Date.fromYMD(year, start_month, 1)
        end_date = ee.Date.fromYMD(year, end_month, 1).advance(1, 'month')

        # Build collection
        collections = []

        if end_date.millis().getInfo() > ee.Date('2013-03-18').millis().getInfo():
            l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                .filterDate(start_date, end_date) \
                .filterBounds(study_area) \
                .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_COVER_MAX)) \
                .select(['SR_B3', 'SR_B6', 'QA_PIXEL'], ['Green', 'SWIR1', 'QA_PIXEL'])
            collections.append(l8)

        l7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
            .filterDate(start_date, end_date) \
            .filterBounds(study_area) \
            .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_COVER_MAX)) \
            .select(['SR_B2', 'SR_B5', 'QA_PIXEL'], ['Green', 'SWIR1', 'QA_PIXEL'])
        collections.append(l7)

        if start_date.millis().getInfo() < ee.Date('2013-06-05').millis().getInfo():
            l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2') \
                .filterDate(start_date, end_date) \
                .filterBounds(study_area) \
                .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_COVER_MAX)) \
                .select(['SR_B2', 'SR_B5', 'QA_PIXEL'], ['Green', 'SWIR1', 'QA_PIXEL'])
            collections.append(l5)

        if not collections:
            return {
                'year': year,
                'bimonth': bimonth,
                'water_area_km2': 0,
                'filename': 'N/A',
                'status': 'no_collections',
                'image_count': 0
            }

        # Merge collections
        landsat_collection = collections[0]
        for col in collections[1:]:
            landsat_collection = landsat_collection.merge(col)

        # Apply cloud mask
        masked_collection = landsat_collection.map(mask_landsat_clouds)

        # Check image count
        count = masked_collection.size().getInfo()
        if count == 0:
            return {
                'year': year,
                'bimonth': bimonth,
                'water_area_km2': 0,
                'filename': 'N/A',
                'status': 'no_images',
                'image_count': 0
            }

        # Create composite
        composite = masked_collection.median().clip(study_area)

        # Calculate MNDWI and water mask
        mndwi = composite.normalizedDifference(['Green', 'SWIR1'])
        water_mask = mndwi.gt(WATER_THRESHOLD).rename('water').toByte()

        # Calculate water area
        pixel_area = water_mask.multiply(ee.Image.pixelArea())
        water_area = pixel_area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=study_area,
            scale=30,
            maxPixels=1e10,
            bestEffort=True
        )

        area_km2 = ee.Number(water_area.get('water', 0)).divide(1e6).getInfo()

        # Skip if no water
        if area_km2 == 0:
            return {
                'year': year,
                'bimonth': bimonth,
                'water_area_km2': 0,
                'filename': 'N/A',
                'status': 'no_water',
                'image_count': count
            }

        # Export with coarser resolution
        out_tif = os.path.join(out_dir, f"water_mask_{year:04d}_{bimonth}.tif")

        # Try export at 60m resolution first
        try:
            geemap.ee_export_image(
                water_mask.selfMask(),
                filename=out_tif,
                scale=EXPORT_SCALE,  # 60m resolution
                region=study_area,
                file_per_band=False,
                crs='EPSG:4326'
            )

            if os.path.exists(out_tif):
                return {
                    'year': year,
                    'bimonth': bimonth,
                    'water_area_km2': area_km2,
                    'filename': os.path.basename(out_tif),
                    'status': 'success',
                    'image_count': count
                }
        except:
            # Fallback: use bounds instead of polygon
            try:
                bounds = study_area.bounds()
                geemap.ee_export_image(
                    water_mask.selfMask(),
                    filename=out_tif,
                    scale=EXPORT_SCALE,
                    region=bounds,
                    file_per_band=False,
                    crs='EPSG:4326'
                )

                if os.path.exists(out_tif):
                    return {
                        'year': year,
                        'bimonth': bimonth,
                        'water_area_km2': area_km2,
                        'filename': os.path.basename(out_tif),
                        'status': 'success',
                        'image_count': count
                    }
            except:
                pass

        return {
            'year': year,
            'bimonth': bimonth,
            'water_area_km2': area_km2,
            'filename': 'N/A',
            'status': 'export_failed',
            'image_count': count
        }

    except Exception as e:
        return {
            'year': year,
            'bimonth': bimonth,
            'water_area_km2': 0,
            'filename': 'N/A',
            'status': f'error: {str(e)[:30]}',
            'image_count': 0
        }

# ======================================================================
#                       MAIN PROCESSING
# ======================================================================

print("="*60)
print(f"Landsat Bi-Monthly Water Mask Export ({START_YEAR}-{END_YEAR})")
print(f"Cloud Cover Max: {CLOUD_COVER_MAX}%")
print(f"Export Scale: {EXPORT_SCALE}m (coarser for file size)")
print("="*60)

# Generate all periods
all_periods = []
for year in range(START_YEAR, END_YEAR + 1):
    for bimonth_id, bimonth_info in BIMONTHS.items():
        all_periods.append({
            'year': year,
            'bimonth': bimonth_id,
            'bimonth_name': bimonth_info['name'],
            'months': bimonth_info['months']
        })

total_periods = len(all_periods)
print(f"\nTotal bi-monthly periods: {total_periods}\n")

# Process each period
export_summary = {
    'successful': 0,
    'failed': 0,
    'total_size_mb': 0,
    'areas': []
}

print("="*60)
print("EXPORTING WATER MASKS...")
print("="*60)

for i, period_info in enumerate(all_periods):
    year = period_info['year']
    bimonth = period_info['bimonth']
    label = f"{year}-{bimonth}"

    out_tif = os.path.join(out_dir, f"water_mask_{year:04d}_{bimonth}.tif")

    # Skip if exists
    if os.path.exists(out_tif):
        file_size_mb = os.path.getsize(out_tif) / (1024 * 1024)
        export_summary['successful'] += 1
        export_summary['total_size_mb'] += file_size_mb
        print(f"[{i+1}/{total_periods}] {label}... ⏭️ Exists ({file_size_mb:.2f} MB)")
        continue

    print(f"[{i+1}/{total_periods}] {label} ({period_info['bimonth_name']})...", end='', flush=True)

    result = export_bimonthly_water_mask(
        year,
        bimonth,
        period_info['months'],
        period_info['bimonth_name']
    )

    if result['status'] == 'success':
        file_size_mb = os.path.getsize(out_tif) / (1024 * 1024)
        export_summary['successful'] += 1
        export_summary['total_size_mb'] += file_size_mb
        export_summary['areas'].append(result['water_area_km2'])
        print(f" ✅ ({result['water_area_km2']:.2f} km², {file_size_mb:.2f} MB, {result['image_count']} imgs)")
    else:
        export_summary['failed'] += 1
        print(f" ❌ {result['status']}")

    time.sleep(2)

# Summary
print("\n" + "="*60)
print("COMPLETE!")
print("="*60)
print(f"✅ Successful: {export_summary['successful']}/{total_periods}")
print(f"❌ Failed: {export_summary['failed']}")
print(f"💾 Total size: {export_summary['total_size_mb']:.2f} MB")

if export_summary['successful'] > 0:
    print(f"📊 Avg per file: {export_summary['total_size_mb']/export_summary['successful']:.2f} MB")

if export_summary['areas']:
    print(f"\n💧 Water Area Statistics:")
    print(f"  • Average: {np.mean(export_summary['areas']):.2f} km²")
    print(f"  • Range: {np.min(export_summary['areas']):.2f} - {np.max(export_summary['areas']):.2f} km²")

print(f"\n📁 Output: {out_dir}")
print("="*60)

Landsat Bi-Monthly Water Mask Export (2000-2014)
Cloud Cover Max: 50%
Export Scale: 60m (coarser for file size)

Total bi-monthly periods: 90

EXPORTING WATER MASKS...
[1/90] 2000-BM1 (Jan-Feb)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2000_BM1.tif
 ✅ (1104.32 km², 0.09 MB, 17 imgs)
[2/90] 2000-BM2 (Mar-Apr)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2000_BM2.tif
 ✅ (514.49 km², 0.04 MB, 5 imgs)
[3/90] 2000-BM3 (May-Jun)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2000_BM3.tif
 ✅ (802.54 km², 0.07 MB, 8 imgs)
[4/90] 2000-BM4 (Jul-Aug)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2000_BM4.tif
 ✅ (1695.43 km², 0.10 MB, 4 imgs)
[5/90] 2000-BM5 (Sep-Oct)...Generating URL ...
Please wait ...
Data downloaded to

Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2001_BM1.tif
 ✅ (1255.45 km², 0.11 MB, 22 imgs)
[8/90] 2001-BM2 (Mar-Apr)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2001_BM2.tif
 ✅ (650.38 km², 0.04 MB, 16 imgs)
[9/90] 2001-BM3 (May-Jun)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2001_BM3.tif
 ✅ (908.21 km², 0.06 MB, 8 imgs)
[10/90] 2001-BM4 (Jul-Aug)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2001_BM4.tif
 ✅ (664.20 km², 0.05 MB, 4 imgs)
[11/90] 2001-BM5 (Sep-Oct)...Generating URL ...
Please wait ...
Data downloaded to /content/Landsat_BiMonthly_WaterMasks_2000-2014/water_mask_2001_BM5.tif
 ✅ (2002.48 km², 0.12 MB, 5 imgs)
[12/90] 2001-BM6 (Nov-Dec)...Generating URL ...
Please wait ...
Data downloaded to /con

In [ ]:
import zipfile

# ======================================================================
#                     ZIP OUTPUT FILES
# ======================================================================

def zip_output_files(output_directory):
    zip_filename = f"{output_directory}.zip"
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        # Walk through the output directory and add files to the ZIP
        for root, dirs, files in os.walk(output_directory):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, arcname=os.path.relpath(file_path, output_directory))

    print(f"\n📦 Created ZIP archive: {zip_filename}")
    return zip_filename

# Call the zip function after the processing
zip_output_files(out_dir)


📦 Created ZIP archive: ./Landsat_BiMonthly_WaterMasks_2000-2014.zip


'./Landsat_BiMonthly_WaterMasks_2000-2014.zip'